# 33 · Reliability drill: kill a replica, read the admission metrics
Level L5 · Part 5 · core · ~25 min · builds on lessons 30, 31

Time: about 25 min. This lesson starts **three SIMULATED vLLM replicas** in Docker Compose (the first start builds the image). They run no model: they take time, apply a queue limit like vLLM's `--max-num-queued-reqs`, and expose vLLM metric names. Without Docker, the tables come from the **recorded output** (`data/recorded/33.json`), labelled as such. The check on the real engine, the pinned image `vllm/vllm-openai-cpu:v0.32.0`, is recorded too: it was run live on the course machine.

Ports: 8330 to 8332 (replicas r1 to r3, container port 8000). RAM: about 1 GB for the three containers.

## Goal
By the end you can kill one replica of a three-node cluster under load, show that the queue limit turns the overload into fast 503 refusals that the client retries, and read `vllm:admission_rejections_total` from `/metrics` on the pinned vLLM version.

## Why it matters
- **Without it:** nothing bounds the queue. A burst fills every replica's waiting line; each request waits behind dozens of others, the client gives up after its own timeout and sends the request again, and the retry joins the same line. The GPU spends its time on answers nobody reads. A dead replica that the gateway keeps sending to fails every call it gets.
- **With it:** the engine refuses a request at the door, with HTTP 503, while the queue is full. The client tries another replica at once. The refusal is counted, so the dashboard shows the overload instead of a silent timeout.
- **When you don't need it:** one replica with a small, steady load that never peaks. A queue limit then only refuses work you could serve.

## The idea
A serving fleet stays up with three parts. A **readiness check** (`/health`) says which replicas may take traffic. A **queue limit** rejects early, so a backlog cannot grow without bound. A **retry on another replica** with a short cooldown finds a healthy one. Losing a node tests all three at once: the survivors take its share of the load, so their queue limit starts to refuse, and the counter shows it. Think of a restaurant with eight seats: when the room is full, the host says "come back in ten minutes" at the door, and the guests do not wait an hour in a line.

## Picture
```mermaid
flowchart LR
  C["Client: 120 requests"] --> G["Gateway: next replica in turn, retry on 503 or a dead node"]
  G --> R1["r1: queue limit 8"]
  G --> R2["r2: killed in the drill"]
  G --> R3["r3: queue limit 8"]
  R1 -. "full: 503, counted in admission_rejections" .-> G
```

## Step by step

### Step 1: the names, and the flag that sets the limit
The table lists the names on `/metrics` that this drill reads. The last row is the counter that the pinned engine's source defines (Step 2).

| Name on `/metrics` | Type | Meaning |
| --- | --- | --- |
| `vllm:num_requests_running` | gauge | requests in the batch |
| `vllm:num_requests_waiting` | gauge | requests waiting for a slot |
| `vllm:kv_cache_usage_perc` | gauge | KV memory in use, 0 to 1 |
| `vllm:request_success_total` | counter | finished requests |
| `vllm:admission_rejections_total` | counter | requests refused by a queue limit, labelled `reason` |

The limit is set by `--max-num-queued-reqs` (requests allowed in the engine: running plus waiting). A request over it is refused with HTTP 503, and the counter gets the reason `max_num_queued_reqs`. The flag `--max-num-queued-tokens` works the same way on tokens.

The setup cell sets the names, the replica addresses and the recorded run. It runs with or without Docker.

In [1]:
import asyncio, atexit, itertools, json, random, shutil, subprocess, time
from pathlib import Path
import httpx
from prometheus_client.parser import text_string_to_metric_families

PROJECT, COMPOSE = "tut-kv-forge-15h-33", "../infra/compose/reliability-3node.yml"
REPLICAS = {"r1": "http://127.0.0.1:8330", "r2": "http://127.0.0.1:8331", "r3": "http://127.0.0.1:8332"}
MODEL, IMAGE = "brightside-qwen-0.5b", "vllm/vllm-openai-cpu:v0.32.0"        # the pinned engine image of lessons 08 to 27
SRC = "/opt/venv/lib/python3.12/site-packages/vllm/v1/metrics/loggers.py"     # where the engine defines the counter
COOLDOWN_S, TRIES, IN_FLIGHT, N_REQ = 5.0, 3, 18, 120
REC = json.loads(Path("../data/recorded/33.json").read_text())                # the recorded runs (Docker-free)
LIVE, RESULTS, PIN_LIVE = False, {}, None
print("lab:", PROJECT, "| recorded run:", REC["lesson"], REC["recorded"])

lab: tut-kv-forge-15h-33 | recorded run: 33 2026-10-23


The helpers come next. `compose` starts and stops the lab. `wait_health` waits for a replica's `/health`. `rejections` reads the counter from a `/metrics` page, by reason.

In [2]:
def compose(*args):
    r = subprocess.run(["docker", "compose", "-p", PROJECT, "-f", COMPOSE, *args], capture_output=True, text=True)
    if r.returncode: raise RuntimeError(r.stderr[-800:])

def wait_health(url, limit=180):
    """Seconds from now until the replica's /health answers 200."""
    t0 = time.perf_counter()
    for _ in range(limit):
        try:
            if httpx.get(url + "/health", timeout=2).status_code == 200: return round(time.perf_counter() - t0, 1)
        except httpx.TransportError: pass
        time.sleep(1)
    raise TimeoutError(url)

def rejections(text):
    """{reason: count} from vllm:admission_rejections_total. The page adds _total; the short name is not on it."""
    return {s.labels["reason"]: int(s.value) for f in text_string_to_metric_families(text)
            for s in f.samples if s.name == "vllm:admission_rejections_total"}
print("helpers ready")

helpers ready


### Step 2: check the counter in the pinned engine
The same counter name is in the source of the pinned image, `vllm/vllm-openai-cpu:v0.32.0`. The cell below greps that source inside the image (a few seconds once the image is on this machine). The recorded run also sent 12 requests at once to a server started with `--max-num-queued-reqs 2`: the engine answered 2 and refused 10.

In [3]:
if shutil.which("docker"):
    r = subprocess.run(["docker", "run", "--rm", "--entrypoint", "grep", IMAGE, "-n", "admission_rejections", SRC],
                       capture_output=True, text=True, timeout=900)
    PIN_LIVE = r.stdout.splitlines() if r.returncode == 0 else None
    print("grep of the pinned image:", len(PIN_LIVE or []), "lines")
else:
    print("Docker not found: Step 2 uses the recorded grep")

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


The output of that check is shown here. Without Docker, it shows the recorded grep and the recorded run.

In [4]:
pin = REC["pinned"]
print("source:", "this machine, the pinned image" if PIN_LIVE else "recorded output")
for line in (PIN_LIVE or pin["grep"]): print(line)
print("\nrecorded run on", pin["image"], "with --max-num-queued-reqs 2 (12 requests at once):")
print("status codes:", pin["status_codes"], "| refusal body:", pin["rejection_body"][0])

print("\n".join(pin["metrics_lines"]))

source: recorded output
847:        counter_admission_rejections = self._counter_cls(
848:            name="vllm:admission_rejections",
854:        self.admission_rejections = {
855:            reason: counter_admission_rejections.labels(

recorded run on vllm/vllm-openai-cpu:v0.32.0 with --max-num-queued-reqs 2 (12 requests at once):
status codes: {'200': 2, '503': 10} | refusal body: {"error":{"message":"The engine is currently busy and cannot accept new requests. Please try again later or on a different instance.","type":"Service Unavailabl
vllm:admission_rejections_total{model_name="brightside-qwen-0.5b",reason="max_num_queued_reqs"} 10.0
vllm:admission_rejections_total{model_name="brightside-qwen-0.5b",reason="max_num_queued_tokens"} 0.0


**Read it.** A refusal has status 503. Lines 847 to 855 of the engine define the counter with a `reason` label (lines 854 and 855 make one series per reason), so the page shows one line per reason. A lookup by the short name fails, as Common mistakes shows.

### Step 3: start the three replicas
The Compose file starts r1 to r3 with a queue limit of 8 each. The cell waits until every replica answers `/health`, then sets `LIVE`. The project name keeps this lab's containers apart from other courses. **To run this live:** set `TUT_ENABLE=docker` before you start Jupyter.

| Setting | Value | Where it is set |
| --- | --- | --- |
| queue limit (running plus waiting) | 8 | `QUEUE_LIMIT` in `reliability-3node.yml` |
| decode slots (requests running at once) | 4 | `SLOTS` in `apps/33/mock_vllm.py` (like `--max-num-seqs 4` on the real engine) |

In [5]:
compose("up", "-d", "--build")
for url in REPLICAS.values(): wait_health(url)
LIVE = True
atexit.register(lambda: LIVE and compose("down", "-v"))
print("three replicas answer /health")

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


### Step 4: the client side: a gateway with a cooldown and a retry
The gateway is written by hand so you can see the rule. It sends each call to the next replica in turn that is not cooling down. A connection error puts that replica in cooldown for five seconds. A 503 is a refusal, so the call goes to another replica after a short backoff.

The workload is fixed (seed 33): 120 requests, each a 200-word policy prompt plus a 20 to 60 word question, 32-token answers.
The client keeps 18 in flight. Over three replicas that is six each, under the limit of eight. Over two it is nine each,
so the ninth request on a replica meets a full queue. The refused call goes to the other survivor, which may be full too.
In the recorded run this gave 8 refusals and 1 failed call; the exact numbers change with timing.

In [6]:
rng = random.Random(33)
POLICY = [f"clause-{i}" for i in range(200)]
WORDS = [POLICY + [f"q{rng.randrange(500)}" for _ in range(rng.randint(20, 60))] for _ in range(N_REQ)]

class Gateway:
    """Next replica in turn that is not cooling down. A dead replica cools down for COOLDOWN_S."""
    def __init__(self):
        self.turn, self.cool = itertools.cycle(REPLICAS), dict.fromkeys(REPLICAS, 0.0)
    def pick(self):
        for _ in range(len(REPLICAS)):
            r = next(self.turn)
            if self.cool[r] <= time.monotonic(): return r
        return None                                           # every replica is cooling down

`send` makes one request and applies the rule: up to three tries, each on the replica the gateway picks. A 200 ends it. A refusal (503) or a dead node moves the call on. `calls` counts every try, retries included.

In [7]:
async def send(client, gw, words, stats):
    body = {"model": MODEL, "max_tokens": 32, "messages": [{"role": "user", "content": " ".join(words)}]}
    for attempt in range(TRIES):
        r = gw.pick()
        if r is None: break
        stats["calls"] += 1
        try:
            resp = await client.post(REPLICAS[r] + "/v1/chat/completions", json=body, timeout=30)
        except httpx.TransportError:                          # the node is gone: cool it down, try another
            gw.cool[r] = time.monotonic() + COOLDOWN_S; stats["conn_error"] += 1; continue
        if resp.status_code == 200:
            stats["ok"] += 1; stats["served"][r] += 1
            if attempt: stats["retried_ok"] += 1
            return
        stats["503"] += 1                                     # queue full: refused at the door
        await asyncio.sleep(0.05 * (attempt + 1))             # short backoff before the next try
    stats["failed"] += 1
print("retry rule: up to", TRIES, "tries, cooldown", COOLDOWN_S, "s, workload", N_REQ, "requests")

retry rule: up to 3 tries, cooldown 5.0 s, workload 120 requests


`counters` reads every reachable replica's rejection counter. `run_phase` sends the 120 requests and, with `kill`, stops that replica after 40 have finished, as a node loss would. It reports the rise of each counter during the phase.

In [8]:
def counters():
    out = {}
    for r, url in REPLICAS.items():
        try: out[r] = rejections(httpx.get(url + "/metrics", timeout=5).text)
        except httpx.TransportError: out[r] = None            # the replica does not answer
    return out

async def run_phase(kill=None):
    stats = {"calls": 0, "ok": 0, "retried_ok": 0, "503": 0, "conn_error": 0, "failed": 0,
             "served": dict.fromkeys(REPLICAS, 0), "killed": None}
    before, gw, gate, done = counters(), Gateway(), asyncio.Semaphore(IN_FLIGHT), [0]
    async with httpx.AsyncClient() as client:
        async def one(words):
            async with gate:
                await send(client, gw, words, stats)
                done[0] += 1
                if kill and done[0] == 40 and stats["killed"] is None:
                    stats["killed"] = kill; await asyncio.to_thread(compose, "stop", kill)
        await asyncio.gather(*(one(w) for w in WORDS))
    after = counters()
    rise = {r: after[r]["max_num_queued_reqs"] - before[r]["max_num_queued_reqs"] if before[r] and after[r] else None
            for r in REPLICAS}                                # None: down before or after
    stats["rejected_by"], stats["rejected"], stats["end"] = rise, sum(v for v in rise.values() if v), after
    stats["r1_lines"] = [l for l in httpx.get(REPLICAS["r1"] + "/metrics", timeout=5).text.splitlines()
                         if l.startswith("vllm:admission_rejections_total")]
    return stats

### Step 5: the drill
Three phases, one after another: all replicas up; r2 killed during the second phase; r2 started again for the third. Two things to know. The "r2 stops after 40 requests" rule counts finished requests across all replicas, so some requests sent to r2 are still in flight when it stops, and they become connection errors. And r2's counter starts again at 0 after the restart, because the replica is a new process: its recovered count is a fresh count. Without Docker the cells do nothing, and Step 6 reads the recording.

In [9]:
if LIVE:
    RESULTS["healthy"] = await run_phase()
    RESULTS["node loss"] = await run_phase(kill="r2")          # r2 stops after 40 requests have finished
    compose("start", "r2"); wait_health(REPLICAS["r2"])
    RESULTS["recovered"] = await run_phase()

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


### Step 6: read the drill
The table comes from this live run if Docker ran, and from the recording otherwise. `calls` counts every try, retries included. `rejected` is the rise of the counter during each phase, summed over the replicas that answered before and after it. `r2` shows what it served in that phase before it stopped.

In [10]:
R = RESULTS if LIVE else REC["phases"]
print("source:", "this live run" if LIVE else "recorded output")
print(f"{'phase':<10} {'calls':>5} {'ok':>4} {'503s':>5} {'retried ok':>10} {'failed':>6}  {'r1':>4} {'r2':>4} {'r3':>4}  {'rejected':>8}")
for k, s in R.items():
    print(f"{k:<10} {s['calls']:>5} {s['ok']:>4} {s['503']:>5} {s['retried_ok']:>10} {s['failed']:>6}  "
          f"{s['served']['r1']:>4} {s['served']['r2']:>4} {s['served']['r3']:>4}  {s['rejected']:>8}")
print("\nrefusals per replica in each phase (the rise of the counter):", R["node loss"]["rejected_by"])
end = R["recovered"]["end"]
print("counters at the end of the drill (a counter only rises; r2 restarted, so it counts from 0):",
      {r: end[r]["max_num_queued_reqs"] for r in end})

source: recorded output
phase      calls   ok  503s retried ok failed    r1   r2   r3  rejected
healthy      120  120     0          0      0    40   40   40         0
node loss    129  119     8          5      1    47   26   46         8
recovered    120  120     0          0      0    40   40   40         0

refusals per replica in each phase (the rise of the counter): {'r1': 4, 'r2': None, 'r3': 4}
counters at the end of the drill (a counter only rises; r2 restarted, so it counts from 0): {'r1': 4, 'r2': 0, 'r3': 4}


**Read it.** These counts come from the three simulated replicas, not from a real engine. Only the Step 2 check ran on real vLLM. Healthy: the limit is not reached, so no refusals. When r2 stops, r1 and r3 each hold about nine requests against a limit of eight, so each refuses its ninth, and the gateway retries it on the other survivor. In the recorded run the node-loss `rejected` column is 8 = 4 + 4 (r1 4, r3 4; the end counters show it). `retried ok` counts the requests a retry rescued (5 in the recording); `failed` counts requests that used all three tries (1 in the recording). A call that finds r2 already gone cools it down (`conn_error`). r2's served count in that phase (26 in the recording) counts only answers given before it stopped. After r2 restarts it counts from 0, and the refusals stop.

### Step 7: stop the cluster
Save the live run as a run-time file (not committed) and stop the containers. Without Docker this cell does nothing.

In [11]:
if LIVE:
    Path("../data/33").mkdir(parents=True, exist_ok=True)
    Path("../data/33/results.json").write_text(json.dumps({"phases": RESULTS}, indent=1))
    print("saved the live run to data/33/results.json (run-time file, not committed)")
    compose("down", "-v"); LIVE = False

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


## At work
- **Queue collapse (RESEARCH.md, section 7, row 4):** a bounded queue that refuses early keeps latency flat under overload; an unbounded one turns overload into timeouts and useless retries. Teams alert on the rate of `vllm:admission_rejections_total`. → lesson 35 (a later lesson; rollouts with an eval gate), where you meet this alert as a rollout check.
- **Node loss (section 7, row 8):** streams on a dead node are lost. Clients retry with an idempotency key, the gateway drains a node before it stops, and a PodDisruptionBudget keeps the last replica up. → lesson 32 (readiness and drain).
- **Token limits (section 7, row 9):** a limit on requests per minute means little when one request is 100 tokens and another 100,000. → lesson 29 (the gateway).
- **Go deeper:** an OOM shows as KV usage near 1.0 and a container OOMKilled; a slow GPU or an NCCL stall needs the engine's debug flags (`NCCL_DEBUG=TRACE`, section 7b).

## Common mistakes
**1. Reading the counter by its short name.** The engine defines `vllm:admission_rejections`, and the page shows `vllm:admission_rejections_total` with a `reason` label. The lookup below uses the short name and fails. It reads the recorded page, so it runs without Docker. **This cell fails on purpose.**

In [12]:
text = "\n".join(R["recovered"]["r1_lines"])
names = {s.name: s.value for f in text_string_to_metric_families(text) for s in f.samples}
print(names["vllm:admission_rejections"])

KeyError: 'vllm:admission_rejections'

**The fix:** the parser keeps the `_total` name, and the reason is a label: read `rejections(text)["max_num_queued_reqs"]`, as `counters()` in Step 4 does. Check the exact names on the page after each engine upgrade.

**2. Retrying a refusal on the same replica.** A 503 means the queue is full now. A retry at once to the same replica meets the same full queue. The gateway moves each retry to another replica and waits a little first. After a request has failed all its tries, the client should wait before it sends the request again (with a budget); an immediate loop adds load to the survivors.

## Production checklist
- Set the queue limit from your own load test (lesson 27): each request inside the limit holds KV memory and a place in line.
- Alert on the rate of `vllm:admission_rejections_total`, not on its total: a counter only rises.
- Size the fleet to survive one lost node at peak: each survivor takes N/(N-1) of the load. Here 18 in flight over three, then over two.
- Give retries a budget and a backoff, so a retry storm does not double the load on the survivors.
- Route only to replicas whose `/health` answers 200, and drain a replica before you stop it (lesson 32).
- Pin the engine version and check the metric names after each upgrade (lessons 30 and 31).

## Try it
From `R`, for each phase compute the share of calls that came back 503 (503s divided by `calls`, which counts every try, retries included), and name the phase where the most calls were refused. Then say what a `failed` request means for the client.

Run the Step 6 cell first: without Docker it fills `R` from the recording, and this exercise reads `R`.

In [13]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [14]:
for k in R:
    print(f"{k:<10} 503 share {R[k]['503'] / R[k]['calls']:.0%}  failed {R[k]['failed']}")
worst = max(R, key=lambda k: R[k]["503"] / R[k]["calls"])
print("most refusals per call:", worst, "| a failed request: every try was refused or cooled down, and the client gets an error")

healthy    503 share 0%  failed 0
node loss  503 share 6%  failed 1
recovered  503 share 0%  failed 0
most refusals per call: node loss | a failed request: every try was refused or cooled down, and the client gets an error


## Recap
- A queue limit refuses a request with HTTP 503 while the queue is full. It keeps latency bounded, and it counts each refusal.
- Losing a node moves its share onto the survivors, and the survivors start to refuse. The gateway retries onto another replica after a cooldown.
- Read `vllm:admission_rejections_total` by its full name and its `reason` label, and read its rate next to the queue depth.
- **Interview:** "How do you keep an LLM serving fleet up when a node dies?" Strong answer: readiness checks take the dead node out of rotation, a queue limit refuses the extra load early with a 503, clients retry on another replica with a backoff and a budget, and you size N minus one replicas for peak and alert on the rejection rate.